# 01. Cloudless DET Feedback GA Search Tutorial

이 노트북은 기존 01의 cell 기능/process를 유지하되, 실제 local 모델 실행 경로를 `00_local_qwen_worker_preflight.ipynb`에서 검증된 canonical model-suite 경로로 교체한 버전입니다.

최종 실행 경로:

```text
utils/ga_search/model_suite_benchmark.py
utils/ga_search/local_model_registry.py
utils/ga_search/local_worker.py
utils/ga_search/candidate_generation.py
utils/ga_search/evaluation.py
```

유지하는 process:

1. canonical import / compile / render smoke
2. single-row strict DET evaluation
3. single-row retry / diagnostic
4. optional worker real row smoke
5. category-level benchmark
6. manual prompt patch application and visibility check
7. prompt logs / raw responses / service context inspection

중요:

- 기본 평가 경로에서 `mock`을 사용하지 않습니다.
- `mock`은 GT를 Generated로 복사할 수 있으므로, 이 노트북의 모델 성능 확인에는 사용하지 않습니다.
- 먼저 `qwen25_coder_7b`로 row-level smoke를 확인하고, 이후 14B를 실행합니다.
- `max_new_tokens=64`는 JSON truncation으로 `invalid_json`이 날 수 있으므로 7B 기본값은 `512`, 14B 기본값은 `1024`입니다.

In [1]:
# ============================================================
# Setup 1: imports and server preset
# ============================================================

import os
import sys
import json
import shlex
import time
import html
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 300)

SERVER_PRESET = os.environ.get("SERVER_PRESET", "a6000")  # "a6000" or "a100"

if SERVER_PRESET == "a6000":
    os.environ.setdefault("JOILANG_BASE_DIR", "/home/mgjeong/Desktop/llm/JOILang-Server")
    os.environ.setdefault("JOI_PY", "/home/mgjeong/miniconda3/envs/joi/bin/python")
    os.environ.setdefault("LOCAL_MODEL_BASE", "/home/mgjeong/Desktop/llm/local_models")
elif SERVER_PRESET == "a100":
    os.environ.setdefault("JOILANG_BASE_DIR", "/root/llm/JOILang-Server")
    # A100/root server keeps the existing root Python unless explicitly overridden.
    os.environ.setdefault("JOI_PY", "/root/miniconda3/bin/python")
    os.environ.setdefault("LOCAL_MODEL_BASE", "/root/llm/local_models")
else:
    raise ValueError(f"Unknown SERVER_PRESET={SERVER_PRESET!r}")

In [2]:
# ============================================================
# Setup 2: paths, canonical helpers, and viewers
# ============================================================

BASE_DIR = Path(os.environ["JOILANG_BASE_DIR"]).expanduser().resolve()
JOI_PY = Path(os.environ["JOI_PY"]).expanduser().resolve()
LOCAL_MODEL_BASE = Path(os.environ["LOCAL_MODEL_BASE"]).expanduser().resolve()

assert BASE_DIR.exists(), f"BASE_DIR not found: {BASE_DIR}"
assert JOI_PY.exists(), f"JOI_PY not found: {JOI_PY}"

GA_CLI = BASE_DIR / "utils" / "ga_search" / "cli.py"
DATASET = BASE_DIR / "datasets" / "JOICommands-280.csv"
SERVICE_SCHEMA = BASE_DIR / "datasets" / "service_list_ver2.0.1.json"

MODEL = os.environ.get("JOI_GA_MODEL", "gpt_mg.version0_13")
ROW_NO = int(os.environ.get("ROW_NO", "251"))

MODEL_KEY_7B = os.environ.get("MODEL_KEY_7B", "qwen25_coder_7b")
MODEL_KEY_14B = os.environ.get("MODEL_KEY_14B", "qwen25_coder_14b")

CUDA_VISIBLE_DEVICES = os.environ.get("CUDA_VISIBLE_DEVICES_FOR_NOTEBOOK", "0")
LOCAL_DEVICE = os.environ.get("LOCAL_DEVICE", "cuda:0")

MAX_NEW_TOKENS_7B = int(os.environ.get("MAX_NEW_TOKENS_7B", "512"))
MAX_NEW_TOKENS_14B = int(os.environ.get("MAX_NEW_TOKENS_14B", "1024"))

CATEGORY_TO_RUN = int(os.environ.get("CATEGORY_TO_RUN", "5"))
LIMIT_PER_CATEGORY = int(os.environ.get("LIMIT_PER_CATEGORY", "5"))

RUN_TAG = os.environ.get("RUN_TAG", datetime.now().strftime("%Y%m%d_%H%M%S"))
NB_ROOT = BASE_DIR / "artifacts" / "ga_search_tutorial_runs" / f"cloudless_model_suite_{RUN_TAG}"
NB_ROOT.mkdir(parents=True, exist_ok=True)

print("SERVER_PRESET:", SERVER_PRESET)
print("BASE_DIR:", BASE_DIR)
print("JOI_PY:", JOI_PY)
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE)
print("NB_ROOT:", NB_ROOT)
print("MODEL:", MODEL)
print("ROW_NO:", ROW_NO)
print("MODEL_KEY_7B:", MODEL_KEY_7B)
print("MODEL_KEY_14B:", MODEL_KEY_14B)
print("CUDA_VISIBLE_DEVICES:", CUDA_VISIBLE_DEVICES)
print("LOCAL_DEVICE:", LOCAL_DEVICE)

def ts():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def out_dir(label):
    p = NB_ROOT / str(label)
    p.mkdir(parents=True, exist_ok=True)
    return p

def run_cmd(cmd, log_path=None, check=False, timeout_sec=None, env_extra=None):
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["CUDA_VISIBLE_DEVICES"] = str(CUDA_VISIBLE_DEVICES)
    # Avoid stale /usr/local/cuda-* LD paths interfering with PyTorch wheel CUDA runtime.
    env["LD_LIBRARY_PATH"] = ""
    if env_extra:
        env.update({str(k): str(v) for k, v in env_extra.items()})

    cmd = [str(x) for x in cmd]
    print("\n[CMD]")
    print(" ".join(shlex.quote(x) for x in cmd))
    if log_path:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        print("[LOG]", log_path)

    started = time.time()
    proc = subprocess.run(
        cmd,
        cwd=str(BASE_DIR),
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout_sec,
    )
    elapsed = time.time() - started
    output = proc.stdout or ""

    if log_path:
        log_path.write_text(output, encoding="utf-8", errors="replace")

    print(output[-8000:])
    print(f"[RC] {proc.returncode}  [elapsed] {elapsed:.2f}s")

    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed rc={proc.returncode}: {' '.join(cmd)}")

    return proc.returncode, output

def load_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

def read_csv_or_empty(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_csv(path)

def show_file(path, max_chars=5000):
    path = Path(path)
    print(path, "exists=", path.exists(), "size=", path.stat().st_size if path.exists() else 0)
    if path.exists():
        text = path.read_text(encoding="utf-8", errors="replace")
        print(text[:max_chars])
        if len(text) > max_chars:
            print(f"\n... truncated {len(text) - max_chars} chars")

def suite_model_dir(root, model_key):
    return Path(root) / str(model_key)

def suite_summary(root):
    p = Path(root) / "suite_summary.json"
    return load_json(p) if p.exists() else {}

def model_summary(root, model_key):
    p = suite_model_dir(root, model_key) / "model_summary.json"
    return load_json(p) if p.exists() else {}

def candidates_df(root, model_key=None):
    root = Path(root)
    if model_key:
        p = suite_model_dir(root, model_key) / "candidates" / "generation_000.csv"
    else:
        p = root / "candidates" / "generation_000.csv"
        if not p.exists():
            matches = list(root.glob("*/candidates/generation_000.csv"))
            p = matches[0] if matches else p
    return read_csv_or_empty(p)

def eval_df(root, model_key=None):
    root = Path(root)
    if model_key:
        p = suite_model_dir(root, model_key) / "eval" / "row_evaluation.csv"
    else:
        p = root / "eval" / "row_evaluation.csv"
        if not p.exists():
            matches = list(root.glob("*/eval/row_evaluation.csv"))
            p = matches[0] if matches else p
    return read_csv_or_empty(p)

def cli_base(command):
    return [str(JOI_PY), "-m", "utils.ga_search.cli", command]

def run_render(label="render_smoke", user_input="Turn on the light.", dry_run=True):
    od = out_dir(label)
    cmd = cli_base("render") + [
        "--model", MODEL,
        "--user-input", user_input,
        "--search-mode", "auto",
    ]
    if dry_run:
        cmd.append("--dry-run")
    rc, output = run_cmd(cmd, log_path=od / f"{label}.log", check=True, timeout_sec=300)
    return od, rc

def model_suite_cmd(output_dir, model_key, row_no=None, max_new_tokens=512, timeout_sec=900, extra_args=None):
    cmd = [
        str(JOI_PY), "-m", "utils.ga_search.model_suite_benchmark",
        "--model", MODEL,
        "--model-key", str(model_key),
        "--llm-mode", "worker",
        "--local-model-base-dir", str(LOCAL_MODEL_BASE),
        "--worker-python", str(JOI_PY),
        "--local-device", str(LOCAL_DEVICE),
        "--local-files-only", "true",
        "--local-trust-remote-code", "true",
        "--local-max-new-tokens", str(max_new_tokens),
        "--timeout-sec", str(timeout_sec),
        "--output-dir", str(output_dir),
    ]
    if row_no is not None:
        cmd += ["--row-no", str(row_no)]
    if extra_args:
        cmd += list(map(str, extra_args))
    return cmd

def run_model_suite_row(label, model_key, row_no, max_new_tokens, timeout_sec=1200, extra_args=None, check=False):
    root = out_dir(label)
    cmd = model_suite_cmd(
        root,
        model_key=model_key,
        row_no=row_no,
        max_new_tokens=max_new_tokens,
        timeout_sec=timeout_sec,
        extra_args=extra_args,
    )
    rc, output = run_cmd(cmd, log_path=root / "run.log", check=check, timeout_sec=timeout_sec + 300)
    return root, rc

def inspect_run(root, model_key, max_rows=20):
    root = Path(root)
    print("\nRUN_ROOT:", root)
    print("suite_summary:")
    print(json.dumps(suite_summary(root), ensure_ascii=False, indent=2)[:4000])
    print("\nmodel_summary:")
    print(json.dumps(model_summary(root, model_key), ensure_ascii=False, indent=2)[:4000])

    cand = candidates_df(root, model_key)
    ev = eval_df(root, model_key)

    print("\nCANDIDATES:")
    display(cand.head(max_rows))
    print("\nEVALUATION:")
    display(ev.head(max_rows))

    if not cand.empty:
        first = cand.iloc[0]
        for col in ["generation_error_type", "candidate_error", "prompt_log_paths", "raw_response_path", "service_context_source", "service_list_retrieval_scores", "repair_actions"]:
            if col in cand.columns:
                print("\n" + "=" * 100)
                print(col)
                print(first.get(col, ""))

        raw_path = first.get("raw_response_path", "")
        if isinstance(raw_path, str) and raw_path.strip():
            print("\n" + "=" * 100)
            print("RAW RESPONSE")
            show_file(raw_path, max_chars=6000)
    return cand, ev

def row_summary(root, model_key=None, max_rows=120):
    ev = eval_df(root, model_key)
    if ev.empty:
        print("No eval rows")
        return ev
    cols = [c for c in [
        "row_no", "category", "genome_id", "candidate_index", "det_score", "det_pass", "gt_exact",
        "gt_similarity", "schedule_match", "service_recall", "service_precision", "receiver_recall",
        "numeric_grounding", "enum_grounding", "dataflow_score", "failure_reasons", "generated_code", "gt_code"
    ] if c in ev.columns]
    display(ev[cols].head(max_rows))
    return ev

def gt_pretty(raw):
    if raw is None:
        return ""
    if not isinstance(raw, str):
        raw = json.dumps(raw, ensure_ascii=False)
    raw = raw.strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict):
            script = obj.get("script", obj.get("code", ""))
            meta = dict(obj)
            meta.pop("script", None)
            meta.pop("code", None)
            return "[JSON meta]\n" + json.dumps(meta, ensure_ascii=False, indent=2) + "\n\n[script/code]\n" + str(script).replace("\\n", "\n")
        return json.dumps(obj, ensure_ascii=False, indent=2)
    except Exception:
        return raw.replace("\\n", "\n").replace("\\t", "    ")

def _pre(title, text):
    return f"""
    <div>
      <div style="font-weight:700;background:#f7f7f7;padding:6px;border:1px solid #ddd;border-bottom:none;">{html.escape(str(title))}</div>
      <pre style="margin:0;white-space:pre;overflow:auto;max-height:520px;min-height:160px;tab-size:4;font-family:Consolas,'Courier New',monospace;font-size:13px;line-height:1.45;background:#fbfbfb;padding:10px;border:1px solid #ddd;">{html.escape(gt_pretty(text))}</pre>
    </div>
    """

def show_gt_vs_generated(root, row_no=None, model_key=None, max_rows=20):
    cand = candidates_df(root, model_key)
    ev = eval_df(root, model_key)
    if cand.empty:
        print("No candidates found:", root)
        return pd.DataFrame()

    if row_no is not None and "row_no" in cand.columns:
        cand = cand[cand["row_no"].astype(str) == str(row_no)]

    if not ev.empty:
        join_cols = [c for c in ["row_no", "genome_id", "candidate_index"] if c in cand.columns and c in ev.columns]
        merged = cand.merge(ev, on=join_cols, how="left", suffixes=("", "_eval")) if join_cols else cand
    else:
        merged = cand

    cards = []
    for _, r in merged.head(max_rows).iterrows():
        gt = r.get("gt", r.get("gt_json", ""))
        generated = r.get("generated_json", "") or r.get("candidates", "") or r.get("generated_code", "")
        title = f"row={r.get('row_no')} cat={r.get('category')} genome={r.get('genome_id')} det={r.get('det_score', '')} pass={r.get('det_pass', '')}"
        cards.append(f"""
        <div style="border:1px solid #ccc;border-radius:8px;padding:12px;margin:12px 0;">
          <div style="font-weight:700;margin-bottom:8px;">{html.escape(str(title))}</div>
          <div style="display:grid;grid-template-columns:minmax(0,1fr) minmax(0,1fr);gap:12px;">
            {_pre("GT", gt)}
            {_pre("Generated", generated)}
          </div>
          <div style="margin-top:8px;font-size:13px;"><b>failure_reasons:</b> {html.escape(str(r.get('failure_reasons', '')))}</div>
          <div style="margin-top:4px;font-size:13px;"><b>prompt_log_paths:</b> {html.escape(str(r.get('prompt_log_paths', '')))}</div>
          <div style="margin-top:4px;font-size:13px;"><b>raw_response_path:</b> {html.escape(str(r.get('raw_response_path', '')))}</div>
        </div>
        """)
    display(HTML("\n".join(cards)))
    return merged

def select_rows_by_category(category, limit_per_category=5):
    ds = pd.read_csv(DATASET)
    cat_col = "category" if "category" in ds.columns else "cat"
    row_col = "row_no" if "row_no" in ds.columns else None
    hit = ds[ds[cat_col].astype(str) == str(category)].copy()
    if row_col:
        rows = hit[row_col].dropna().astype(int).tolist()
    else:
        rows = [int(i) + 1 for i in hit.index.tolist()]
    return rows[: int(limit_per_category)]

def run_category_rows(label, model_key, category, limit_per_category, max_new_tokens, timeout_sec=1200, extra_args=None):
    root = out_dir(label)
    rows = select_rows_by_category(category, limit_per_category)
    print("CATEGORY ROWS:", rows)
    records = []
    for row_no in rows:
        sub_root, rc = run_model_suite_row(
            label=f"{label}_row{row_no:03d}_{ts()}",
            model_key=model_key,
            row_no=row_no,
            max_new_tokens=max_new_tokens,
            timeout_sec=timeout_sec,
            extra_args=extra_args,
            check=False,
        )
        records.append({"row_no": row_no, "rc": rc, "run_root": str(sub_root)})
    idx = pd.DataFrame(records)
    idx_path = root / "category_run_index.csv"
    idx.to_csv(idx_path, index=False)
    print("category index:", idx_path)
    display(idx)
    return root, records

def show_category_results(records, model_key):
    cand_frames = []
    eval_frames = []
    for r in records:
        rr = Path(r["run_root"])
        cdf = candidates_df(rr, model_key)
        edf = eval_df(rr, model_key)
        if not cdf.empty:
            cdf["run_root"] = str(rr)
            cand_frames.append(cdf)
        if not edf.empty:
            edf["run_root"] = str(rr)
            eval_frames.append(edf)
    cand = pd.concat(cand_frames, ignore_index=True) if cand_frames else pd.DataFrame()
    ev = pd.concat(eval_frames, ignore_index=True) if eval_frames else pd.DataFrame()
    print("CATEGORY CANDIDATES:")
    display(cand.head(80))
    print("CATEGORY EVAL:")
    display(ev.head(80))
    return cand, ev

def show_artifact_table(paths):
    display(pd.DataFrame([
        {
            "path": str(p),
            "exists": Path(p).exists(),
            "size": Path(p).stat().st_size if Path(p).exists() else 0,
        }
        for p in paths
    ]))

def run_patch_apply(label, patches_path, genome_json=None):
    od = out_dir(label)
    cmd = cli_base("patch-apply") + [
        "--model", MODEL,
        "--prompt-patches", str(patches_path),
        "--out-dir", str(od),
    ]
    if genome_json:
        cmd += ["--genome-json", str(genome_json)]
    rc, out = run_cmd(cmd, log_path=od / f"{label}.log", check=True, timeout_sec=300)
    return od

def compare_eval_runs(before_root, after_root, model_key):
    b = eval_df(before_root, model_key)
    a = eval_df(after_root, model_key)
    if b.empty or a.empty:
        print("Missing eval outputs")
        return pd.DataFrame()
    keys = [k for k in ["row_no", "genome_id", "candidate_index"] if k in b.columns and k in a.columns]
    if not keys:
        keys = ["row_no"]
    merged = b.merge(a, on=keys, how="outer", suffixes=("_before", "_after"))
    display(merged.head(120))
    return merged

SERVER_PRESET: a6000
BASE_DIR: /home/mgjeong/Desktop/llm/JOILang-Server
JOI_PY: /home/mgjeong/miniconda3/envs/joi/bin/python3.10
LOCAL_MODEL_BASE: /home/mgjeong/Desktop/llm/local_models
NB_ROOT: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001
MODEL: gpt_mg.version0_13
ROW_NO: 251
MODEL_KEY_7B: qwen25_coder_7b
MODEL_KEY_14B: qwen25_coder_14b
CUDA_VISIBLE_DEVICES: 0
LOCAL_DEVICE: cuda:0


## 1. Canonical import / compile / render smoke

In [3]:
import_check = r"""
import importlib.util
for name in [
    "utils.ga_search.cli",
    "utils.ga_search.model_resolver",
    "utils.ga_search.render_adapter",
    "utils.ga_search.candidate_generation",
    "utils.ga_search.evaluation",
    "utils.ga_search.ga_engine",
    "utils.ga_search.local_model_registry",
    "utils.ga_search.local_worker",
    "utils.ga_search.model_suite_benchmark",
    "utils.det_evaluator",
]:
    spec = importlib.util.find_spec(name)
    print(name, "=>", spec.origin if spec else None)
    assert spec is not None
    forbidden = "version0_15" + "_update20260413"
    assert forbidden not in str(spec.origin)
"""
run_cmd([str(JOI_PY), "-c", import_check], log_path=out_dir("import_origin_check") / "import_origin_check.log", check=True, timeout_sec=120)
run_cmd([str(JOI_PY), "-m", "compileall", "utils/ga_search", "utils/det_evaluator.py"], log_path=out_dir("compileall_utils") / "compileall.log", check=True, timeout_sec=300)
_ = run_render(label="render_smoke", user_input="Turn on the light.", dry_run=True)


[CMD]
/home/mgjeong/miniconda3/envs/joi/bin/python3.10 -c '
import importlib.util
for name in [
    "utils.ga_search.cli",
    "utils.ga_search.model_resolver",
    "utils.ga_search.render_adapter",
    "utils.ga_search.candidate_generation",
    "utils.ga_search.evaluation",
    "utils.ga_search.ga_engine",
    "utils.ga_search.local_model_registry",
    "utils.ga_search.local_worker",
    "utils.ga_search.model_suite_benchmark",
    "utils.det_evaluator",
]:
    spec = importlib.util.find_spec(name)
    print(name, "=>", spec.origin if spec else None)
    assert spec is not None
    forbidden = "version0_15" + "_update20260413"
    assert forbidden not in str(spec.origin)
'
[LOG] /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/import_origin_check/import_origin_check.log
utils.ga_search.cli => /home/mgjeong/Desktop/llm/JOILang-Server/utils/ga_search/cli.py
utils.ga_search.model_resolver => /home/mgjeong/Desktop/llm/JOIL

## 2. Single-row strict DET eval: real local Qwen 7B worker

기존 01의 single-row strict DET 확인 기능을 유지하되, `mock` 대신 00에서 검증한 `utils.ga_search.model_suite_benchmark` worker 경로를 사용합니다.

In [4]:
ROW_NO = int(os.environ.get("ROW_NO", str(ROW_NO)))

print("ACTIVE ROW_NO:", ROW_NO)
print("ACTIVE MODEL_KEY_7B:", MODEL_KEY_7B)
print("ACTIVE MAX_NEW_TOKENS_7B:", MAX_NEW_TOKENS_7B)
print("ACTIVE JOI_PY:", JOI_PY)
print("ACTIVE LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE)

qwen7b_root, qwen7b_rc = run_model_suite_row(
    label=f"qwen7b_row{ROW_NO}_{ts()}",
    model_key=MODEL_KEY_7B,
    row_no=ROW_NO,
    max_new_tokens=MAX_NEW_TOKENS_7B,
    timeout_sec=1200,
    check=False,
)
print("qwen7b_rc:", qwen7b_rc)

qwen7b_cand, qwen7b_eval = inspect_run(qwen7b_root, MODEL_KEY_7B)
_ = show_gt_vs_generated(qwen7b_root, row_no=ROW_NO, model_key=MODEL_KEY_7B, max_rows=10)

qwen7b_ms = model_summary(qwen7b_root, MODEL_KEY_7B) or {}
qwen7b_generation_error_rate = float(qwen7b_ms.get("generation_error_rate", 1.0) or 1.0)
if qwen7b_generation_error_rate != 0.0:
    print("\n[WARN] Qwen 7B generation_error_rate != 0. Inspect raw response above.")
else:
    print("\n[OK] Real local Qwen 7B generation completed without generation error.")

ACTIVE ROW_NO: 251
ACTIVE MODEL_KEY_7B: qwen25_coder_7b
ACTIVE MAX_NEW_TOKENS_7B: 512
ACTIVE JOI_PY: /home/mgjeong/miniconda3/envs/joi/bin/python3.10
ACTIVE LOCAL_MODEL_BASE: /home/mgjeong/Desktop/llm/local_models

[CMD]
/home/mgjeong/miniconda3/envs/joi/bin/python3.10 -m utils.ga_search.model_suite_benchmark --model gpt_mg.version0_13 --model-key qwen25_coder_7b --llm-mode worker --local-model-base-dir /home/mgjeong/Desktop/llm/local_models --worker-python /home/mgjeong/miniconda3/envs/joi/bin/python3.10 --local-device cuda:0 --local-files-only true --local-trust-remote-code true --local-max-new-tokens 512 --timeout-sec 1200 --output-dir /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/qwen7b_row251_20260624_015006 --row-no 251
[LOG] /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/qwen7b_row251_20260624_015006/run.log
{
  "output_root": "/home/mgjeong/Deskto

,row_no,category,command_eng,command_kor,gt,genome_id,generation,candidate_index,candidate_strategy,prompt_render_mode,prompt_assets_dir,prompt_log_paths,raw_response_path,candidates,generated_code,generated_json,generation_error_type,generation_error_count,generation_prompt_tokens_total,generation_completion_tokens_total,generation_total_tokens_total,latency_sec,peak_vram_gb,service_context_source,service_context_mode,service_list_retrieval_scores,connected_devices_used,service_schema_path,repair_applied,repair_actions,backend
0,251,8,"At midnight, close the door and check the light every hour until 6 AM; if the brightness is greater than 30, lower it to 10.","자정이 되면 문을 닫고, 오전 6시까지 한 시간마다 조명을 체크해서 밝기가 30보다 크면 10으로 낮춰줘.","{""name"": """", \n""cron"": ""0 0 * * *"", \n""period"": 3600000, \n""script"": ""\nactive := 0\n\nif (active == 0) {\n\n (#Door).door_close()\n\n active = 1\n\n}\n\nif ((#Clock).clock_hour == 6) {\n\n break\n\n}\n\nif ((#Light).lightsensor_brightness > 30) {\n\n (#Light).light_movetobrightness(...",suite_base,0,0,worker_direct,monolith,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_13,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/qwen7b_row251_20260624_015006/qwen25_coder_7b/prompts/row_251_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_2...",/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/qwen7b_row251_20260624_015006/qwen25_coder_7b/raw_responses/row_251_cand_0_gen_000.json,"[{""name"": ""MidnightDoorAndLightControl"", ""cron"": ""0 0 * * *"", ""period"": 3600000, ""code"": ""active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(1...","active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(10, 10, 10)\n }\n} while ((#Clock).clock_hour < 6)","{""name"": ""MidnightDoorAndLightControl"", ""cron"": ""0 0 * * *"", ""period"": 3600000, ""code"": ""active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(10...",NaN,0,53156,138,0,29.744723,24.1082,provided_schema,schema_fallback,"{""status"": ""retrieval_disabled"", ""reason"": ""canonical ga_search has not enabled service retrieval yet""}",NaN,datasets/service_list_ver2.0.1.json,False,[],worker



EVALUATION:


,row_no,category,genome_id,candidate_index,det_score,det_pass,gt_exact,gt_similarity,schedule_match,cron_match,period_match,code_match,service_valid,receiver_valid,gt_service_coverage,gt_service_precision,gt_receiver_coverage,dataflow_score,numeric_grounding,enum_grounding,failure_reasons,diff_summary,gt_code,generated_code,gt_json,generated_json,generation,candidate_strategy,generation_error_type,generation_error_count,generation_prompt_tokens_total,generation_completion_tokens_total,generation_total_tokens_total,latency_sec,peak_vram_gb,raw_response_path,prompt_log_paths,service_schema_path,service_context_mode,service_context_source,service_list_retrieval_scores,repair_applied,repair_actions,backend
0,251,8,suite_base,0,52.0533,False,False,0.360111,True,True,True,False,False,False,0.25,0.25,0.5,1.0,0.5,1.0,"[""gt_mismatch"",""gt_service_coverage"",""unknown_service"",""gt_receiver_coverage"",""numeric_grounding""]","{""gt_services"":[""clock_hour"",""door_close"",""light_movetobrightness"",""lightsensor_brightness""],""generated_services"":[""clock_hour"",""doorControl_close"",""lightLevel_light"",""light_moveToRGB""],""gt_receivers"":[""#Door"",""#Light"",""(#Clock"",""(#Light""],""generated_receivers"":[""#DoorLock"",""#Light"",""#LightSenso...",\nactive := 0\n\nif (active == 0) {\n\n (#Door).door_close()\n\n active = 1\n\n}\n\nif ((#Clock).clock_hour == 6) {\n\n break\n\n}\n\nif ((#Light).lightsensor_brightness > 30) {\n\n (#Light).light_movetobrightness(10)\n\n},"active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(10, 10, 10)\n }\n} while ((#Clock).clock_hour < 6)","{""name"":"""",""cron"":""0 0 * * *"",""period"":3600000,""script"":""\nactive := 0\n\nif (active == 0) {\n\n (#Door).door_close()\n\n active = 1\n\n}\n\nif ((#Clock).clock_hour == 6) {\n\n break\n\n}\n\nif ((#Light).lightsensor_brightness > 30) {\n\n (#Light).light_movetobrightness(10)\n\n}""}","{""name"":""MidnightDoorAndLightControl"",""cron"":""0 0 * * *"",""period"":3600000,""code"":""active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(10, 10, 1...",0,worker_direct,NaN,0,53156,138,0,29.744723,24.1082,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/qwen7b_row251_20260624_015006/qwen25_coder_7b/raw_responses/row_251_cand_0_gen_000.json,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/qwen7b_row251_20260624_015006/qwen25_coder_7b/prompts/row_251_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_2...",datasets/service_list_ver2.0.1.json,schema_fallback,provided_schema,"{""status"": ""retrieval_disabled"", ""reason"": ""canonical ga_search has not enabled service retrieval yet""}",False,[],worker



generation_error_type
nan

prompt_log_paths
["/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/qwen7b_row251_20260624_015006/qwen25_coder_7b/prompts/row_251_cand_0_gen_000.md", "/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/qwen7b_row251_20260624_015006/qwen25_coder_7b/prompts/row_251_cand_0_gen_000.json"]

raw_response_path
/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/qwen7b_row251_20260624_015006/qwen25_coder_7b/raw_responses/row_251_cand_0_gen_000.json

service_context_source
provided_schema

service_list_retrieval_scores
{"status": "retrieval_disabled", "reason": "canonical ga_search has not enabled service retrieval yet"}

repair_actions
[]

RAW RESPONSE
/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/qwen7b_row251_20260


[WARN] Qwen 7B generation_error_rate != 0. Inspect raw response above.


## 3. Single-row strict DET eval: retry / diagnostic

기존 01의 non-mock diagnostic 기능을 유지하되, local endpoint unavailable test가 아니라 00에서 검증한 worker 경로의 longer-output retry로 바꿉니다.

In [5]:
RUN_QWEN7B_LONG_RETRY = False
MAX_NEW_TOKENS_7B_LONG = int(os.environ.get("MAX_NEW_TOKENS_7B_LONG", "1024"))

if RUN_QWEN7B_LONG_RETRY:
    qwen7b_long_root, qwen7b_long_rc = run_model_suite_row(
        label=f"qwen7b_row{ROW_NO}_long_{ts()}",
        model_key=MODEL_KEY_7B,
        row_no=ROW_NO,
        max_new_tokens=MAX_NEW_TOKENS_7B_LONG,
        timeout_sec=5000,
        check=False,
    )
    print("qwen7b_long_rc:", qwen7b_long_rc)
    inspect_run(qwen7b_long_root, MODEL_KEY_7B)
    _ = show_gt_vs_generated(qwen7b_long_root, row_no=ROW_NO, model_key=MODEL_KEY_7B, max_rows=10)
else:
    print("Long retry skipped. Set RUN_QWEN7B_LONG_RETRY=True only if 512-token run returns invalid_json or truncated JSON.")

Long retry skipped. Set RUN_QWEN7B_LONG_RETRY=True only if 512-token run returns invalid_json or truncated JSON.


## 4. Optional worker real row smoke

기존 01의 optional worker real smoke 기능을 유지하되, `utils.ga_search.cli search` skeleton이 아니라 model-suite worker row benchmark로 14B를 실행합니다.

In [6]:
RUN_QWEN14B_AFTER_7B_OK = os.environ.get("RUN_QWEN14B_AFTER_7B_OK", "false").lower() == "true"

if RUN_QWEN14B_AFTER_7B_OK and qwen7b_generation_error_rate == 0.0:
    qwen14b_root, qwen14b_rc = run_model_suite_row(
        label=f"qwen14b_row{ROW_NO}_{ts()}",
        model_key=MODEL_KEY_14B,
        row_no=ROW_NO,
        max_new_tokens=MAX_NEW_TOKENS_14B,
        timeout_sec=5000,
        check=False,
    )
    print("qwen14b_rc:", qwen14b_rc)
    inspect_run(qwen14b_root, MODEL_KEY_14B)
    _ = show_gt_vs_generated(qwen14b_root, row_no=ROW_NO, model_key=MODEL_KEY_14B, max_rows=10)
elif RUN_QWEN14B_AFTER_7B_OK:
    print("Qwen 14B skipped because Qwen 7B had generation errors.")
else:
    print("Qwen 14B skipped. Set RUN_QWEN14B_AFTER_7B_OK=true after Qwen 7B succeeds.")

Qwen 14B skipped. Set RUN_QWEN14B_AFTER_7B_OK=true after Qwen 7B succeeds.


## 5. Category search: real local Qwen worker by repeated row-level benchmark

기존 01의 category-level 확인 기능을 유지하되, mock search 대신 category row들을 선택해서 model-suite worker row benchmark를 반복합니다.

In [7]:
RUN_CATEGORY_7B = os.environ.get("RUN_CATEGORY_7B", "true").lower() == "true"
RUN_CATEGORY_14B = os.environ.get("RUN_CATEGORY_14B", "false").lower() == "true"

category_7b_records = []
category_14b_records = []

if RUN_CATEGORY_7B:
    category_7b_root, category_7b_records = run_category_rows(
        label=f"category{CATEGORY_TO_RUN}_qwen7b_{ts()}",
        model_key=MODEL_KEY_7B,
        category=CATEGORY_TO_RUN,
        limit_per_category=LIMIT_PER_CATEGORY,
        max_new_tokens=MAX_NEW_TOKENS_7B,
        timeout_sec=5000,
    )
    category_7b_candidates, category_7b_eval = show_category_results(category_7b_records, MODEL_KEY_7B)
else:
    print("Category 7B run skipped. Set RUN_CATEGORY_7B=true to run.")

if RUN_CATEGORY_14B:
    category_14b_root, category_14b_records = run_category_rows(
        label=f"category{CATEGORY_TO_RUN}_qwen14b_{ts()}",
        model_key=MODEL_KEY_14B,
        category=CATEGORY_TO_RUN,
        limit_per_category=LIMIT_PER_CATEGORY,
        max_new_tokens=MAX_NEW_TOKENS_14B,
        timeout_sec=2400,
    )
    category_14b_candidates, category_14b_eval = show_category_results(category_14b_records, MODEL_KEY_14B)
else:
    print("Category 14B run skipped. Set RUN_CATEGORY_14B=true after 7B category run is stable.")

CATEGORY ROWS: [121, 122, 123, 124, 125]

[CMD]
/home/mgjeong/miniconda3/envs/joi/bin/python3.10 -m utils.ga_search.model_suite_benchmark --model gpt_mg.version0_13 --model-key qwen25_coder_7b --llm-mode worker --local-model-base-dir /home/mgjeong/Desktop/llm/local_models --worker-python /home/mgjeong/miniconda3/envs/joi/bin/python3.10 --local-device cuda:0 --local-files-only true --local-trust-remote-code true --local-max-new-tokens 512 --timeout-sec 5000 --output-dir /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/category5_qwen7b_20260624_015123_row121_20260624_015123 --row-no 121
[LOG] /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/category5_qwen7b_20260624_015123_row121_20260624_015123/run.log
{
  "output_root": "/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/category5_qwen7b_20260624_01

,row_no,rc,run_root
0,121,0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/category5_qwen7b_20260624_015123_row121_20260624_015123
1,122,0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/category5_qwen7b_20260624_015123_row122_20260624_015151
2,123,0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/category5_qwen7b_20260624_015123_row123_20260624_015220
3,124,0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/category5_qwen7b_20260624_015123_row124_20260624_015248
4,125,0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/category5_qwen7b_20260624_015123_row125_20260624_015315


CATEGORY CANDIDATES:


,row_no,category,command_eng,command_kor,gt,genome_id,generation,candidate_index,candidate_strategy,prompt_render_mode,prompt_assets_dir,prompt_log_paths,raw_response_path,candidates,generated_code,generated_json,generation_error_type,generation_error_count,generation_prompt_tokens_total,generation_completion_tokens_total,generation_total_tokens_total,latency_sec,peak_vram_gb,service_context_source,service_context_mode,service_list_retrieval_scores,connected_devices_used,service_schema_path,repair_applied,repair_actions,backend,run_root
0,121,5,Switch the TV to channel 7 and switch to channel 11 after 1 hour.,TV 채널을 7번으로 바꾸고 1시간 뒤에 11번으로 바꿔줘.,"{""name"": """", \n""cron"": """", \n""period"": 0, \n""script"": ""\n(#Television).television_setchannel(7)\n\ndelay(1 HOUR)\n\n(#Television).television_setchannel(11)""}",suite_base,0,0,worker_direct,monolith,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_13,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/category5_qwen7b_20260624_015123_row121_20260624_015123/qwen25_coder_7b/prompts/row_121_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_ru...",/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/category5_qwen7b_20260624_015123_row121_20260624_015123/qwen25_coder_7b/raw_responses/row_121_cand_0_gen_000.json,"[{""name"": ""TVChannelChange"", ""cron"": """", ""period"": -1, ""code"": ""if ((#Television).tvChannel_tvChannel != 7) {\n (#Television).tvChannel_setTvChannel(7)\n}\ndelay(1 HOUR)\nif ((#Television).tvChannel_tvChannel != 11) {\n (#Television).tvChannel_setTvChannel(11)\n}""}]",if ((#Television).tvChannel_tvChannel != 7) {\n (#Television).tvChannel_setTvChannel(7)\n}\ndelay(1 HOUR)\nif ((#Television).tvChannel_tvChannel != 11) {\n (#Television).tvChannel_setTvChannel(11)\n},"{""name"": ""TVChannelChange"", ""cron"": """", ""period"": -1, ""code"": ""if ((#Television).tvChannel_tvChannel != 7) {\n (#Television).tvChannel_setTvChannel(7)\n}\ndelay(1 HOUR)\nif ((#Television).tvChannel_tvChannel != 11) {\n (#Television).tvChannel_setTvChannel(11)\n}""}",NaN,0,53141,104,0,28.390878,24.1057,provided_schema,schema_fallback,"{""status"": ""retrieval_disabled"", ""reason"": ""canonical ga_search has not enabled service retrieval yet""}",NaN,datasets/service_list_ver2.0.1.json,False,[],worker,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/category5_qwen7b_20260624_015123_row121_20260624_015123
1,122,5,Set the air conditioner to cool mode and switch to auto mode after 30 minutes.,에어컨을 냉방 모드로 설정하고 30분 뒤에 자동 모드로 설정해줘.,"{""name"": """", \n""cron"": """", \n""period"": 0, \n""script"": ""\n(#AirConditioner).airconditioner_setairconditionermode(\""cool\"")\n\ndelay(30 MIN)\n\n(#AirConditioner).airconditioner_setairconditionermode(\""auto\"")""}",suite_base,0,0,worker_direct,monolith,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_13,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/category5_qwen7b_20260624_015123_row122_20260624_015151/qwen25_coder_7b/prompts/row_122_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_ru...",/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/category5_qwen7b_20260624_015123_row122_20260624_015151/qwen25_coder_7b/raw_responses/row_122_cand_0_gen_000.json,"[{""name"": ""AirConditionerCoolAndAuto"", ""cron"": """", ""period"": 1800000, ""code"": ""if ((#AirConditioner).airConditionerMode_airConditionerMode == \""auto\"") {\n (#AirConditioner).airConditionerMode_setAirConditionerMode(\""cool\"")\n} else {\n (#AirConditioner).airConditionerMode_setAirConditionerMod...","if ((#AirConditioner).airConditionerMode_airConditionerMode == ""auto"") 

CATEGORY EVAL:


,row_no,category,genome_id,candidate_index,det_score,det_pass,gt_exact,gt_similarity,schedule_match,cron_match,period_match,code_match,service_valid,receiver_valid,gt_service_coverage,gt_service_precision,gt_receiver_coverage,dataflow_score,numeric_grounding,enum_grounding,failure_reasons,diff_summary,gt_code,generated_code,gt_json,generated_json,generation,candidate_strategy,generation_error_type,generation_error_count,generation_prompt_tokens_total,generation_completion_tokens_total,generation_total_tokens_total,latency_sec,peak_vram_gb,raw_response_path,prompt_log_paths,service_schema_path,service_context_mode,service_context_source,service_list_retrieval_scores,repair_applied,repair_actions,backend,run_root
0,121,5,suite_base,0,51.5377,False,False,0.467925,False,True,False,False,False,True,0.0,0.0,1.0,1.0,1.000000,1.0,"[""period_mismatch"",""gt_mismatch"",""gt_service_coverage"",""unknown_service""]","{""gt_services"":[""television_setchannel""],""generated_services"":[""tvChannel_setTvChannel"",""tvChannel_tvChannel""],""gt_receivers"":[""#Television""],""generated_receivers"":[""#Television"",""(#Television""],""gt_numeric_literals"":[""7"",""1"",""11""],""generated_numeric_literals"":[""7"",""7"",""1"",""11"",""11""],""gt_string_...",\n(#Television).television_setchannel(7)\n\ndelay(1 HOUR)\n\n(#Television).television_setchannel(11),if ((#Television).tvChannel_tvChannel != 7) {\n (#Television).tvChannel_setTvChannel(7)\n}\ndelay(1 HOUR)\nif ((#Television).tvChannel_tvChannel != 11) {\n (#Television).tvChannel_setTvChannel(11)\n},"{""name"":"""",""cron"":"""",""period"":0,""script"":""\n(#Television).television_setchannel(7)\n\ndelay(1 HOUR)\n\n(#Television).television_setchannel(11)""}","{""name"":""TVChannelChange"",""cron"":"""",""period"":-1,""code"":""if ((#Television).tvChannel_tvChannel != 7) {\n (#Television).tvChannel_setTvChannel(7)\n}\ndelay(1 HOUR)\nif ((#Television).tvChannel_tvChannel != 11) {\n (#Television).tvChannel_setTvChannel(11)\n}""}",0,worker_direct,NaN,0,53141,104,0,28.390878,24.1057,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/category5_qwen7b_20260624_015123_row121_20260624_015123/qwen25_coder_7b/raw_responses/row_121_cand_0_gen_000.json,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/category5_qwen7b_20260624_015123_row121_20260624_015123/qwen25_coder_7b/prompts/row_121_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_ru...",datasets/service_list_ver2.0.1.json,schema_fallback,provided_schema,"{""status"": ""retrieval_disabled"", ""reason"": ""canonical ga_search has not enabled service retrieval yet""}",False,[],worker,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/category5_qwen7b_20260624_015123_row121_20260624_015123
1,122,5,suite_base,0,30.2873,False,False,0.092910,False,True,False,False,False,True,0.0,0.0,1.0,1.0,0.000000,1.0,"[""period_mismatch"",""gt_mismatch"",""gt_service_coverage"",""unknown_service"",""numeric_grounding""]","{""gt_services"":[""airconditioner_setairconditionermode""],""generated_services"":[""airConditionerMode_airConditionerMode"",""airConditionerMode_setAirConditionerMode""],""gt_receivers"":[""#AirConditioner""],""generated_receivers"":[""#AirConditioner"",""(#AirConditioner""],""gt_numeric_literals"":[""30""],""generate...","\n(#AirConditioner).airconditioner_setairconditionermode(""cool"")\n\ndelay(30 MIN)\n\n(#AirConditioner).airconditioner_setairconditionermode(""auto"")","if ((#AirConditioner).airConditionerMode_airConditionerMode == ""auto"") {\n (#AirConditioner).airConditionerMode_setAirConditionerMode(""cool"")\n} else {\n (#AirConditioner).airConditionerMode_setAirConditionerMode(""cool"")\n (#AirConditioner).airConditionerMode_setAirConditionerMode(""auto"")\n}","{""name"":"""",""cron"":"""",""period"":0,""scri

Category 14B run skipped. Set RUN_CATEGORY_14B=true after 7B category run is stable.


## 6. Manual prompt patch application and visibility check

기존 01의 manual prompt patch 생성/적용/visibility check 기능을 유지합니다. Patch 적용 후 model-suite가 `--genome-json`을 지원하면 optional patched rerun을 수행합니다.

In [10]:
patch_dir = out_dir(f"manual_patch_{ts()}")
patch_path = patch_dir / "prompt_patches.json"
patch_payload = {
    "advisor_meta": {
        "source": "tutorial_manual_patch",
        "official_metric": "strict_det",
        "target_runtime": "utils.ga_search.model_suite_benchmark",
    },
    "prompt_patches": [
        {
            "patch_id": "manual_det_grounding_rule_001",
            "target_block_family": "DET_Helper",
            "target_block_id": "06",
            "operation": "append_micro_rule",
            "priority": 90,
            "patch_text": "Before final JSON, verify schedule, receiver tags, service coverage, numeric/enum grounding, and temporal order against the command.",
            "evidence_rows": [str(ROW_NO)],
            "evidence_failure_reasons": ["gt_mismatch"],
            "mutation_intent": "strict_det_repair"
        }
    ],
}
patch_path.write_text(json.dumps(patch_payload, ensure_ascii=False, indent=2), encoding="utf-8")
show_file(patch_path)

patch_apply_dir = run_patch_apply(f"patch_apply_{ts()}", patch_path)
show_artifact_table([
    patch_apply_dir / "patched_genome.json",
    patch_apply_dir / "patch_application_report.json",
    patch_apply_dir / "patch_diff.md",
    patch_apply_dir / "patched_prompt_preview.md",
])
show_file(patch_apply_dir / "patch_application_report.json")
show_file(patch_apply_dir / "patch_diff.md", max_chars=3000)

RUN_PATCHED_RERUN = os.environ.get("RUN_PATCHED_RERUN", "false").lower() == "true"
patched_7b_root = None
patched_7b_rc = None

if RUN_PATCHED_RERUN:
    patched_genome = patch_apply_dir / "patched_genome.json"
    if not patched_genome.exists():
        raise FileNotFoundError(patched_genome)
    patched_7b_root, patched_7b_rc = run_model_suite_row(
        label=f"qwen7b_row{ROW_NO}_patched_{ts()}",
        model_key=MODEL_KEY_7B,
        row_no=ROW_NO,
        max_new_tokens=MAX_NEW_TOKENS_7B,
        timeout_sec=1200,
        extra_args=["--genome-json", str(patched_genome)],
        check=False,
    )
    print("patched_7b_rc:", patched_7b_rc)
    inspect_run(patched_7b_root, MODEL_KEY_7B)
    _ = compare_eval_runs(qwen7b_root, patched_7b_root, MODEL_KEY_7B)
else:
    print("Patched rerun skipped. Set RUN_PATCHED_RERUN=true if the current model_suite_benchmark supports --genome-json.")

/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/manual_patch_20260624_015619/prompt_patches.json exists= True size= 707
{
  "advisor_meta": {
    "source": "tutorial_manual_patch",
    "official_metric": "strict_det",
    "target_runtime": "utils.ga_search.model_suite_benchmark"
  },
  "prompt_patches": [
    {
      "patch_id": "manual_det_grounding_rule_001",
      "target_block_family": "DET_Helper",
      "target_block_id": "06",
      "operation": "append_micro_rule",
      "priority": 90,
      "patch_text": "Before final JSON, verify schedule, receiver tags, service coverage, numeric/enum grounding, and temporal order against the command.",
      "evidence_rows": [
        "251"
      ],
      "evidence_failure_reasons": [
        "gt_mismatch"
      ],
      "mutation_intent": "strict_det_repair"
    }
  ]
}

[CMD]
/home/mgjeong/miniconda3/envs/joi/bin/python3.10 -m utils.ga_search.cli patch-apply --model gpt_mg.v

RuntimeError: command failed rc=2: /home/mgjeong/miniconda3/envs/joi/bin/python3.10 -m utils.ga_search.cli patch-apply --model gpt_mg.version0_13 --prompt-patches /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/manual_patch_20260624_015619/prompt_patches.json --out-dir /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015001/patch_apply_20260624_015619

## 7. Inspect prompt logs / raw responses / service context

기존 01의 artifact inspection 기능을 유지하되, patched mock search가 아니라 실제 model-suite worker 결과를 확인합니다.

In [9]:
inspect_root = None
inspect_model_key = MODEL_KEY_7B

if patched_7b_root is not None:
    inspect_root = patched_7b_root
elif "qwen7b_root" in globals():
    inspect_root = qwen7b_root
elif category_7b_records:
    inspect_root = Path(category_7b_records[0]["run_root"])

print("INSPECT_ROOT:", inspect_root)
if inspect_root is None:
    print("No model-suite run available to inspect.")
else:
    cdf = candidates_df(inspect_root, inspect_model_key)
    display(cdf.head(20))
    for col in ["generation_error_type", "prompt_log_paths", "raw_response_path", "service_context_source", "service_list_retrieval_scores", "repair_actions"]:
        if col in cdf.columns:
            print("\nCOLUMN:", col)
            print(cdf[col].head(5).to_string(index=False))

    if not cdf.empty:
        first = cdf.iloc[0]
        prompt_paths_raw = first.get("prompt_log_paths", "")
        try:
            prompt_paths = json.loads(prompt_paths_raw) if isinstance(prompt_paths_raw, str) else prompt_paths_raw
        except Exception:
            prompt_paths = []
        if prompt_paths:
            print("\n" + "=" * 100)
            print("FIRST PROMPT LOG")
            show_file(prompt_paths[0], max_chars=6000)

        raw_path = first.get("raw_response_path", "")
        if isinstance(raw_path, str) and raw_path.strip():
            print("\n" + "=" * 100)
            print("RAW RESPONSE")
            show_file(raw_path, max_chars=6000)

print("\nFINAL RUN INDEX")
run_index = []
for name in ["qwen7b_root", "qwen7b_long_root", "qwen14b_root", "category_7b_root", "category_14b_root", "patch_apply_dir", "patched_7b_root"]:
    if name in globals() and globals()[name] is not None:
        run_index.append({"name": name, "path": str(globals()[name]), "exists": Path(globals()[name]).exists()})
display(pd.DataFrame(run_index))
print("NB_ROOT:", NB_ROOT)

NameError: name 'patched_7b_root' is not defined